In [ ]:
# ============================================================
# CELLULE 1 — Connexion Drive + Vérification existant
# ============================================================
# On monte Drive et on vérifie ce qui existe déjà
# pour éviter de refaire un travail déjà fait
# ou de laisser des anciennes données qui pourraient
# interférer avec le nouveau pipeline
# ============================================================

from google.colab import drive
import os

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/projet_chiens/emotions'

# Dossiers qu'on va créer
DOSSIERS = {
    'raw'     : f'{BASE}/raw',        # datasets originaux
    'clean'   : f'{BASE}/clean',      # après nettoyage
    'aug'     : f'{BASE}/augmented',  # après GAN
    'final'   : f'{BASE}/final',      # après split
    'models'  : f'{BASE}/models',     # modèles GAN sauvegardés
}

print("✅ Drive monté !\n")
print("📂 Vérification des dossiers existants :\n")

for nom, chemin in DOSSIERS.items():
    if os.path.exists(chemin):
        # Compter les fichiers
        total = sum(
            len([f for f in files if f.endswith(('.jpg','.jpeg','.png'))])
            for _, _, files in os.walk(chemin)
        )
        print(f"  ⚠️  {nom:10s} → EXISTS ({total} images)")
    else:
        print(f"  ✅ {nom:10s} → absent (sera créé)")

print(f"\n📁 Base : {BASE}")

Mounted at /content/drive
✅ Drive monté !

📂 Vérification des dossiers existants :

  ⚠️  raw        → EXISTS (0 images)
  ⚠️  clean      → EXISTS (28285 images)
  ⚠️  aug        → EXISTS (28280 images)
  ⚠️  final      → EXISTS (28280 images)
  ⚠️  models     → EXISTS (0 images)

📁 Base : /content/drive/MyDrive/projet_chiens/emotions


In [ ]:
# ============================================================
# CELLULE 2 — Imports + Paramètres
# ============================================================
# On importe toutes les bibliothèques nécessaires
# et on définit les paramètres du pipeline
#
# Paramètres clés :
#   IMG_SIZE = 224  → taille standard pour EfficientNet/ResNet/YOLO
#   SEED = 42       → reproductibilité des résultats
#   MIN_SIZE = 50   → on supprime les images < 50×50 pixels
# ============================================================

import os
import shutil
import hashlib
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# ── Chemins ──────────────────────────────────────────────────
BASE      = '/content/drive/MyDrive/projet_chiens/emotions'
RAW_DIR   = f'{BASE}/raw'
CLEAN_DIR = f'{BASE}/clean'
AUG_DIR   = f'{BASE}/augmented'
FINAL_DIR = f'{BASE}/final'
MODEL_DIR = f'{BASE}/models'
CSV_PATH  = f'{BASE}/labels.csv'

# ── Paramètres ────────────────────────────────────────────────
IMG_SIZE  = 224    # taille des images pour les modèles CNN
MIN_SIZE  = 50     # taille minimale acceptable (pixels)
SEED      = 42     # graine aléatoire pour reproductibilité
random.seed(SEED)

# ── Mapping labels originaux → groupe ─────────────────────────
# On garde les 7 labels originaux pour la Late Fusion
# Le groupe (normal/anormal) sera utilisé pour la décision finale
LABEL_TO_GROUPE = {
    'happy'   : 'normal',   # émotion positive ✅
    'relaxed' : 'normal',   # émotion positive ✅
    'neutral' : 'normal',   # émotion neutre ✅
    'sad'     : 'normal',   # ambigu → normal (Late Fusion corrige)
    'frown'   : 'normal',   # ambigu → normal (Late Fusion corrige)
    'alert'   : 'anormal',  # signe de danger ❌
    'angry'   : 'anormal',  # signe d'agressivité ❌
}

# Créer les dossiers
for d in [RAW_DIR, CLEAN_DIR, AUG_DIR, FINAL_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Imports OK")
print(f"📁 Base : {BASE}")
print(f"\n📋 Mapping des {len(LABEL_TO_GROUPE)} labels :")
for label, groupe in LABEL_TO_GROUPE.items():
    print(f"   {label:10s} → {groupe}")

✅ Imports OK
📁 Base : /content/drive/MyDrive/projet_chiens/emotions

📋 Mapping des 7 labels :
   happy      → normal
   relaxed    → normal
   neutral    → normal
   sad        → normal
   frown      → normal
   alert      → anormal
   angry      → anormal


In [ ]:
# ============================================================
# CELLULE 3 — Exploration des datasets existants sur Drive
# ============================================================
# On vérifie exactement ce qu'on a dans raw/
# pour connaître les labels disponibles par dataset
# ============================================================

import os

OLD_RAW = '/content/drive/MyDrive/projet_chiens/datasets/raw'

# Chemins exacts des 4 datasets
DATASETS = {
    'ds1_dfe'        : f'{OLD_RAW}/dfe_dataset/DFE dataset',
    'ds2_dog_emotion': f'{OLD_RAW}/dog_emotion/Dog Emotion',
    'ds3'            : None,  # on va détecter
    'ds4'            : None,  # on va détecter
}

print("📂 Contenu de raw/ :\n")
for ds_name in sorted(os.listdir(OLD_RAW)):
    ds_path = os.path.join(OLD_RAW, ds_name)
    if not os.path.isdir(ds_path):
        continue
    print(f"  📁 {ds_name}/")
    # Chercher les sous-dossiers
    for sub1 in sorted(os.listdir(ds_path)):
        sub1_path = os.path.join(ds_path, sub1)
        if not os.path.isdir(sub1_path):
            continue
        # Chercher les labels
        for sub2 in sorted(os.listdir(sub1_path)):
            sub2_path = os.path.join(sub1_path, sub2)
            if os.path.isdir(sub2_path):
                n = len([f for f in os.listdir(sub2_path)
                         if f.lower().endswith(('.jpg','.jpeg','.png'))])
                print(f"       → {sub2:15s} : {n} images")
    print()

📂 Contenu de raw/ :

  📁 dfe_dataset/
       → branches        : 0 images
       → hooks           : 0 images
       → info            : 0 images
       → logs            : 0 images
       → objects         : 0 images
       → refs            : 0 images
       → angry           : 1000 images
       → happy           : 1000 images
       → neutral         : 1000 images
       → sad             : 1000 images

  📁 dog_emotion/
       → angry           : 1000 images
       → happy           : 1000 images
       → relaxed         : 1000 images
       → sad             : 1000 images

  📁 dog_emotions_5class/
       → alert           : 1865 images
       → angry           : 1865 images
       → frown           : 1865 images
       → happy           : 1865 images
       → relax           : 1865 images

  📁 dog_emotions_pred/
       → angry           : 2256 images
       → happy           : 4784 images
       → relaxed         : 4349 images
       → sad             : 4532 images



In [ ]:
# ============================================================
# CELLULE 4 — Collecte des images par label original
# ============================================================
# On regroupe toutes les images par label
# en combinant les différents datasets
#
# Note : "relax" et "relaxed" sont le même label
# → on les fusionne sous "relaxed"
# ============================================================

import os

OLD_RAW = '/content/drive/MyDrive/projet_chiens/datasets/raw'

# Sources par label (label → liste de dossiers)
SOURCES = {
    'happy'   : [
        f'{OLD_RAW}/dfe_dataset/DFE dataset/happy',
        f'{OLD_RAW}/dog_emotion/Dog Emotion/happy',
        f'{OLD_RAW}/dog_emotions_5class/train_images_5_class/happy',
        f'{OLD_RAW}/dog_emotions_pred/images/happy',
    ],
    'angry'   : [
        f'{OLD_RAW}/dfe_dataset/DFE dataset/angry',
        f'{OLD_RAW}/dog_emotion/Dog Emotion/angry',
        f'{OLD_RAW}/dog_emotions_5class/train_images_5_class/angry',
        f'{OLD_RAW}/dog_emotions_pred/images/angry',
    ],
    'sad'     : [
        f'{OLD_RAW}/dfe_dataset/DFE dataset/sad',
        f'{OLD_RAW}/dog_emotion/Dog Emotion/sad',
        f'{OLD_RAW}/dog_emotions_pred/images/sad',
    ],
    'relaxed' : [
        f'{OLD_RAW}/dog_emotion/Dog Emotion/relaxed',
        f'{OLD_RAW}/dog_emotions_5class/train_images_5_class/relax',
        f'{OLD_RAW}/dog_emotions_pred/images/relaxed',
    ],
    'neutral' : [
        f'{OLD_RAW}/dfe_dataset/DFE dataset/neutral',
    ],
    'alert'   : [
        f'{OLD_RAW}/dog_emotions_5class/train_images_5_class/alert',
    ],
    'frown'   : [
        f'{OLD_RAW}/dog_emotions_5class/train_images_5_class/frown',
    ],
}

# Collecter tous les chemins
images_par_label = {}

print("📊 Comptage par label :\n")
total = 0
for label, dossiers in SOURCES.items():
    chemins = []
    for d in dossiers:
        if os.path.exists(d):
            for f in os.listdir(d):
                if f.lower().endswith(('.jpg','.jpeg','.png')):
                    chemins.append(os.path.join(d, f))
        else:
            print(f"  ⚠️  INTROUVABLE : {d}")

    images_par_label[label] = chemins
    groupe = LABEL_TO_GROUPE[label]
    print(f"  {'✅' if len(chemins) > 0 else '❌'} "
          f"{label:10s} → {groupe:7s} → {len(chemins):5d} images")
    total += len(chemins)

print(f"\n  TOTAL : {total} images")
print(f"  MOYENNE : {total // len(SOURCES)} images par label")

📊 Comptage par label :

  ✅ happy      → normal  →  8649 images
  ✅ angry      → anormal →  6121 images
  ✅ sad        → normal  →  6532 images
  ✅ relaxed    → normal  →  7214 images
  ✅ neutral    → normal  →  1000 images
  ✅ alert      → anormal →  1865 images
  ✅ frown      → normal  →  1865 images

  TOTAL : 33246 images
  MOYENNE : 4749 images par label


In [ ]:
# ============================================================
# CELLULE 5 — Nettoyage RAPIDE (sans ouvrir chaque image)
# ============================================================
# On vérifie seulement la taille du fichier
# au lieu d'ouvrir chaque image avec PIL
# → beaucoup plus rapide ✅
#
# On supprime :
#   → fichiers trop petits (< 1KB = probablement corrompus)
#   → fichiers vides (0 bytes)
# ============================================================
import os

MIN_FILE_SIZE = 1024  # 1 KB minimum

print("🔍 Nettoyage rapide en cours...\n")

total_supprimes = 0

for label, chemins in images_par_label.items():
    supprimes = 0
    valides   = []

    for fpath in chemins:
        try:
            size = os.path.getsize(fpath)
            if size < MIN_FILE_SIZE:
                supprimes += 1
            else:
                valides.append(fpath)
        except:
            supprimes += 1

    images_par_label[label] = valides
    total_supprimes += supprimes
    status = "⚠️ " if supprimes > 0 else "✅"
    print(f"  {status} {label:10s} → {len(valides):5d} valides "
          f"| {supprimes} supprimées")

print(f"\n  Total supprimées : {total_supprimes}")
print("✅ Nettoyage terminé !")

🔍 Nettoyage rapide en cours...

  ✅ happy      →  8649 valides | 0 supprimées
  ✅ angry      →  6121 valides | 0 supprimées
  ✅ sad        →  6532 valides | 0 supprimées
  ✅ relaxed    →  7214 valides | 0 supprimées
  ✅ neutral    →  1000 valides | 0 supprimées
  ✅ alert      →  1865 valides | 0 supprimées
  ✅ frown      →  1865 valides | 0 supprimées

  Total supprimées : 0
✅ Nettoyage terminé !


In [ ]:
# ============================================================
# CELLULE 6 — Déduplication MD5
# ============================================================
# On supprime les images en double entre les datasets
# grâce au hash MD5 — chaque image a une empreinte unique
# Si 2 images ont le même hash → même image → on supprime
#
# Pourquoi c'est important ?
# → les datasets se chevauchent parfois
# → les doublons causent de l'overfitting
# → le modèle mémorise au lieu d'apprendre
# ============================================================
import hashlib

def md5_hash(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

print("🔍 Déduplication MD5 en cours...\n")

total_doublons = 0
hashes_globaux = {}  # hash → premier fichier vu (tous labels confondus)

for label, chemins in images_par_label.items():
    doublons = 0
    valides  = []

    for fpath in chemins:
        try:
            h = md5_hash(fpath)
            if h in hashes_globaux:
                doublons += 1
            else:
                hashes_globaux[h] = fpath
                valides.append(fpath)
        except:
            doublons += 1

    images_par_label[label] = valides
    total_doublons += doublons
    status = "⚠️ " if doublons > 0 else "✅"
    print(f"  {status} {label:10s} → {len(valides):5d} restantes "
          f"| {doublons} doublons supprimés")

print(f"\n  Total doublons : {total_doublons}")
print("✅ Déduplication terminée !")

🔍 Déduplication MD5 en cours...

  ⚠️  happy      →  7034 restantes | 1615 doublons supprimés
  ⚠️  angry      →  4141 restantes | 1980 doublons supprimés
  ⚠️  sad        →  5981 restantes | 551 doublons supprimés
  ⚠️  relaxed    →  4645 restantes | 2569 doublons supprimés
  ⚠️  neutral    →   999 restantes | 1 doublons supprimés
  ⚠️  alert      →     0 restantes | 1865 doublons supprimés
  ⚠️  frown      →     0 restantes | 1865 doublons supprimés

  Total doublons : 10446
✅ Déduplication terminée !


In [ ]:
# ============================================================
# CELLULE 6 CORRIGÉE — Déduplication MD5 par label
# ============================================================
# On déduplique SÉPARÉMENT par label
# → pas de suppression entre labels différents
# → on évite de perdre des classes entières
# ============================================================
import hashlib

def md5_hash(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

# Recharger les chemins originaux
# car images_par_label a été modifié
for label, dossiers in SOURCES.items():
    chemins = []
    for d in dossiers:
        if os.path.exists(d):
            for f in os.listdir(d):
                if f.lower().endswith(('.jpg','.jpeg','.png')):
                    chemins.append(os.path.join(d, f))
    images_par_label[label] = chemins

print("🔍 Déduplication MD5 par label en cours...\n")

total_doublons = 0

for label, chemins in images_par_label.items():
    hashes_vus = {}  # hash local à chaque label
    doublons   = 0
    valides    = []

    for fpath in chemins:
        try:
            h = md5_hash(fpath)
            if h in hashes_vus:
                doublons += 1
            else:
                hashes_vus[h] = fpath
                valides.append(fpath)
        except:
            doublons += 1

    images_par_label[label] = valides
    total_doublons += doublons
    status = "⚠️ " if doublons > 0 else "✅"
    print(f"  {status} {label:10s} → {len(valides):5d} restantes "
          f"| {doublons} doublons supprimés")

print(f"\n  Total doublons : {total_doublons}")
print("✅ Déduplication terminée !")

🔍 Déduplication MD5 par label en cours...

  ⚠️  happy      →  7034 restantes | 1615 doublons supprimés
  ⚠️  angry      →  4665 restantes | 1456 doublons supprimés
  ⚠️  sad        →  6244 restantes | 288 doublons supprimés
  ⚠️  relaxed    →  5687 restantes | 1527 doublons supprimés
  ✅ neutral    →  1000 restantes | 0 doublons supprimés
  ⚠️  alert      →  1863 restantes | 2 doublons supprimés
  ⚠️  frown      →  1859 restantes | 6 doublons supprimés

  Total doublons : 4894
✅ Déduplication terminée !


In [ ]:
# ============================================================
# CELLULE 7 — Resize 224×224 + Sauvegarde sur Drive
# ============================================================
# On redimensionne toutes les images à 224×224 pixels
# car c'est la taille standard pour EfficientNet,
# ResNet et YOLOv8 (pré-entraînés sur ImageNet)
#
# On sauvegarde dans CLEAN_DIR sur Drive pour ne pas
# perdre le travail si la session coupe
# ============================================================
from PIL import Image
import os, shutil

print(f"📐 Resize 224×224 + Sauvegarde dans CLEAN_DIR...\n")

for label, chemins in images_par_label.items():
    dest_dir = os.path.join(CLEAN_DIR, label)
    os.makedirs(dest_dir, exist_ok=True)

    n_ok    = 0
    n_erreur = 0

    for i, fpath in enumerate(chemins):
        try:
            img  = Image.open(fpath).convert('RGB')
            img  = img.resize((224, 224))
            dst  = os.path.join(dest_dir, f'{label}_{i:05d}.jpg')
            img.save(dst, 'JPEG', quality=95)
            n_ok += 1
        except:
            n_erreur += 1

    print(f"  ✅ {label:10s} → {n_ok:5d} images | {n_erreur} erreurs")

print(f"\n✅ Resize terminé !")
print(f"📁 Sauvegardé dans : {CLEAN_DIR}")

📐 Resize 224×224 + Sauvegarde dans CLEAN_DIR...

  ✅ happy      →  7034 images | 0 erreurs
  ✅ angry      →  4665 images | 0 erreurs
  ✅ sad        →  6244 images | 0 erreurs
  ✅ relaxed    →  5687 images | 0 erreurs
  ✅ neutral    →  1000 images | 0 erreurs
  ✅ alert      →  1863 images | 0 erreurs
  ✅ frown      →  1859 images | 0 erreurs

✅ Resize terminé !
📁 Sauvegardé dans : /content/drive/MyDrive/projet_chiens/emotions/clean


In [ ]:
# ============================================================
# CELLULE 8 — Calcul du TARGET
# ============================================================
# On calcule la moyenne du nombre d'images par label
# pour déterminer la cible d'augmentation
#
# Justification mathématique :
# → TARGET = moyenne des classes
# → les classes en dessous sont augmentées
# → les classes au dessus sont undersampleées
# → dataset équilibré ✅
# ============================================================

total  = sum(len(chemins) for chemins in images_par_label.values())
TARGET = total // len(images_par_label)

print("📊 Bilan avant augmentation :\n")
for label, chemins in images_par_label.items():
    groupe = LABEL_TO_GROUPE[label]
    status = "🔧 augmenter" if len(chemins) < TARGET else "✂️  undersample"
    print(f"  {status} | {label:10s} → {groupe:7s} → {len(chemins):5d} images")

print(f"\n  TOTAL   : {total} images")
print(f"  MOYENNE : {total} ÷ {len(images_par_label)} = {TARGET}")
print(f"  TARGET  : {TARGET} images par label")

📊 Bilan avant augmentation :

  ✂️  undersample | happy      → normal  →  7034 images
  ✂️  undersample | angry      → anormal →  4665 images
  ✂️  undersample | sad        → normal  →  6244 images
  ✂️  undersample | relaxed    → normal  →  5687 images
  🔧 augmenter | neutral    → normal  →  1000 images
  🔧 augmenter | alert      → anormal →  1863 images
  🔧 augmenter | frown      → normal  →  1859 images

  TOTAL   : 28352 images
  MOYENNE : 28352 ÷ 7 = 4050
  TARGET  : 4050 images par label


In [ ]:
# ============================================================
# Vérification — Modèles GAN existants sur Drive
# ============================================================
import os

# Chercher les modèles GAN sauvegardés
chemins_gan = [
    '/content/drive/MyDrive/projet_chiens/models/gans',
    '/content/drive/MyDrive/projet_chiens/emotions/models',
    '/content/drive/MyDrive/projet_chiens/datasets/emotions_gan',
]

print("🔍 Recherche des modèles GAN sur Drive...\n")

for chemin in chemins_gan:
    if os.path.exists(chemin):
        contenu = os.listdir(chemin)
        print(f"  ✅ TROUVÉ : {chemin}")
        for f in contenu:
            print(f"       → {f}")
    else:
        print(f"  ❌ Absent : {chemin}")

🔍 Recherche des modèles GAN sur Drive...

  ❌ Absent : /content/drive/MyDrive/projet_chiens/models/gans
  ✅ TROUVÉ : /content/drive/MyDrive/projet_chiens/emotions/models
  ❌ Absent : /content/drive/MyDrive/projet_chiens/datasets/emotions_gan


In [ ]:
# ============================================================
# SAUVEGARDE — État actuel avant changement GPU
# ============================================================
import json

# Sauvegarder les comptages sur Drive
infos = {
    'TARGET' : 4050,
    'labels' : {
        label: len(chemins)
        for label, chemins in images_par_label.items()
    }
}

with open(f'{BASE}/infos.json', 'w') as f:
    json.dump(infos, f, indent=2)

print("✅ Infos sauvegardées !")
print(infos)

✅ Infos sauvegardées !
{'TARGET': 4050, 'labels': {'happy': 7034, 'angry': 4665, 'sad': 6244, 'relaxed': 5687, 'neutral': 1000, 'alert': 1863, 'frown': 1859}}


In [ ]:
# ============================================================
# RECHARGEMENT — Depuis CLEAN_DIR (après redémarrage)
# ============================================================
# Les images sont déjà nettoyées et redimensionnées
# sur Drive → on les charge directement
# sans refaire le nettoyage et la déduplication
# ============================================================

images_par_label = {}

print("📂 Chargement depuis CLEAN_DIR...\n")

for label in LABEL_TO_GROUPE.keys():
    dossier = os.path.join(CLEAN_DIR, label)
    if os.path.exists(dossier):
        chemins = sorted([
            os.path.join(dossier, f)
            for f in os.listdir(dossier)
            if f.lower().endswith('.jpg')
        ])
        images_par_label[label] = chemins
        groupe = LABEL_TO_GROUPE[label]
        print(f"  ✅ {label:10s} → {groupe:7s} → {len(chemins):5d} images")
    else:
        print(f"  ❌ {label:10s} → CLEAN_DIR absent !")

total  = sum(len(v) for v in images_par_label.values())
TARGET = total // len(images_par_label)
print(f"\n  TOTAL  : {total} images")
print(f"  TARGET : {TARGET} images par label")

📂 Chargement depuis CLEAN_DIR...

  ✅ happy      → normal  →  7034 images
  ✅ relaxed    → normal  →  5687 images
  ✅ neutral    → normal  →  1000 images
  ✅ sad        → normal  →  6244 images
  ✅ frown      → normal  →  1792 images
  ✅ alert      → anormal →  1863 images
  ✅ angry      → anormal →  4665 images

  TOTAL  : 28285 images
  TARGET : 4040 images par label


In [ ]:
# ============================================================
# CELLULE 9 — Architecture DCGAN
# ============================================================
# DCGAN = Deep Convolutional Generative Adversarial Network
#
# 2 réseaux s'affrontent :
#   Generator     → crée de fausses images à partir de bruit
#   Discriminator → distingue vraies images / fausses images
#
# Paramètres :
#   LATENT_DIM = 100  → taille du vecteur de bruit en entrée
#   IMG_SIZE   = 64   → on génère en 64×64 (plus rapide)
#                       puis on redimensionne en 224×224
#   CHANNELS   = 3    → images RGB
# ============================================================
import torch
import torch.nn as nn

LATENT_DIM = 100
GAN_SIZE   = 64    # taille de génération
CHANNELS   = 3

# ── Generator ────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            # Entrée : vecteur bruit (100,)
            nn.ConvTranspose2d(LATENT_DIM, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            # 4×4
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # 8×8
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # 16×16
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # 32×32
            nn.ConvTranspose2d(64, CHANNELS, 4, 2, 1, bias=False),
            nn.Tanh()
            # 64×64
        )

    def forward(self, z):
        return self.model(z.view(-1, LATENT_DIM, 1, 1))

# ── Discriminator ─────────────────────────────────────────────
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            # 64×64
            nn.Conv2d(CHANNELS, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 32×32
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 16×16
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # 8×8
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # 4×4
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x).view(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ DCGAN architecture définie")
print(f"   Device    : {device}")
print(f"   Latent dim: {LATENT_DIM}")
print(f"   Image size: {GAN_SIZE}×{GAN_SIZE}")

✅ DCGAN architecture définie
   Device    : cuda
   Latent dim: 100
   Image size: 64×64


In [ ]:
# ============================================================
# CELLULE 10 — Fonction d'entraînement DCGAN
# ============================================================
# On entraîne un GAN séparément pour chaque classe
# qui a besoin d'augmentation (neutral, alert, frown)
#
# Paramètres d'entraînement :
#   EPOCHS    = 100  → nombre de passages sur le dataset
#   LR        = 0.0002 → learning rate standard pour GAN
#   BETA1     = 0.5    → paramètre Adam optimisé pour GAN
#   BATCH_SIZE = 64   → nombre d'images par batch
#
# Pourquoi ces valeurs ?
#   LR=0.0002 et BETA1=0.5 sont les valeurs recommandées
#   dans le papier original DCGAN (Radford et al. 2015)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os

EPOCHS     = 100
LR         = 0.0002
BETA1      = 0.5
BATCH_SIZE = 64

class SimpleDataset(Dataset):
    def __init__(self, chemins, transform):
        self.chemins   = chemins
        self.transform = transform

    def __len__(self):
        return len(self.chemins)

    def __getitem__(self, idx):
        img = Image.open(self.chemins[idx]).convert('RGB')
        return self.transform(img)

# Transformation pour le GAN (64×64, normalisation [-1, 1])
gan_transform = transforms.Compose([
    transforms.Resize((GAN_SIZE, GAN_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)  # [-1, 1]
])

def train_gan(label, chemins, n_generate):
    """
    Entraîne un DCGAN sur les images d'un label
    et génère n_generate nouvelles images
    """
    print(f"\n🔧 GAN pour '{label}' — {len(chemins)} images réelles")
    print(f"   Objectif : générer {n_generate} images\n")

    # Dataset et DataLoader
    dataset = SimpleDataset(chemins, gan_transform)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=2,
                        drop_last=True)

    # Créer Generator et Discriminator
    G = Generator().to(device)
    D = Discriminator().to(device)

    # Optimiseurs
    opt_G = optim.Adam(G.parameters(), lr=LR,
                       betas=(BETA1, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=LR,
                       betas=(BETA1, 0.999))

    criterion = nn.BCELoss()

    # Entraînement
    for epoch in range(EPOCHS):
        for real_imgs in loader:
            real_imgs = real_imgs.to(device)
            bs        = real_imgs.size(0)

            # Labels
            real_labels = torch.ones(bs).to(device)
            fake_labels = torch.zeros(bs).to(device)

            # ── Train Discriminator ──────────────────────
            D.zero_grad()
            out_real = D(real_imgs)
            loss_real = criterion(out_real, real_labels)

            z         = torch.randn(bs, LATENT_DIM).to(device)
            fake_imgs = G(z)
            out_fake  = D(fake_imgs.detach())
            loss_fake = criterion(out_fake, fake_labels)

            loss_D = loss_real + loss_fake
            loss_D.backward()
            opt_D.step()

            # ── Train Generator ──────────────────────────
            G.zero_grad()
            out_fake2 = D(fake_imgs)
            loss_G    = criterion(out_fake2, real_labels)
            loss_G.backward()
            opt_G.step()

        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1:3d}/{EPOCHS} | "
                  f"Loss D: {loss_D.item():.4f} | "
                  f"Loss G: {loss_G.item():.4f}")

    # Sauvegarder le générateur
    os.makedirs(MODEL_DIR, exist_ok=True)
    torch.save(G.state_dict(),
               f'{MODEL_DIR}/generator_{label}.pth')
    print(f"\n  💾 Générateur sauvegardé !")

    # Générer les images
    G.eval()
    dest_dir = os.path.join(AUG_DIR, label)
    os.makedirs(dest_dir, exist_ok=True)

    n_done = 0
    with torch.no_grad():
        while n_done < n_generate:
            bs_gen = min(64, n_generate - n_done)
            z      = torch.randn(bs_gen, LATENT_DIM).to(device)
            imgs   = G(z)
            imgs   = (imgs * 0.5 + 0.5).clamp(0, 1)

            for i in range(bs_gen):
                img_pil = transforms.ToPILImage()(imgs[i].cpu())
                img_pil = img_pil.resize((224, 224))
                img_pil.save(f'{dest_dir}/{label}_gan_{n_done:05d}.jpg',
                             quality=95)
                n_done += 1

    print(f"  ✅ {n_done} images générées dans {dest_dir}")
    return G

print("✅ Fonction train_gan définie")

✅ Fonction train_gan définie


In [ ]:
# ============================================================
# CELLULE 11 — Entraînement GAN pour neutral, alert, frown
# ============================================================
# On entraîne un GAN séparément pour chaque classe
# qui a moins d'images que le TARGET (4040)
#
# Ordre d'entraînement :
#   1. neutral → besoin de 3040 images générées
#   2. alert   → besoin de 2177 images générées
#   3. frown   → besoin de 2248 images générées
#
# Durée estimée : 30-45 min par classe avec GPU T4
# ============================================================

TARGET = 4040

# Classes à augmenter
classes_a_augmenter = {
    label: chemins
    for label, chemins in images_par_label.items()
    if len(chemins) < TARGET
}

print("🚀 Entraînement DCGAN en cours...\n")
print(f"Classes à augmenter : {list(classes_a_augmenter.keys())}\n")

generateurs = {}

for label, chemins in classes_a_augmenter.items():
    n_generate = TARGET - len(chemins)
    print(f"{'='*50}")
    print(f"Label : {label} | À générer : {n_generate} images")
    print(f"{'='*50}")
    G = train_gan(label, chemins, n_generate)
    generateurs[label] = G
    print(f"\n✅ {label} terminé !\n")

print("\n🎉 Tous les GANs entraînés !")
print(f"📁 Images générées dans : {AUG_DIR}")
print(f"📁 Modèles sauvegardés dans : {MODEL_DIR}")

🚀 Entraînement DCGAN en cours...

Classes à augmenter : ['neutral', 'frown', 'alert']

Label : neutral | À générer : 3040 images

🔧 GAN pour 'neutral' — 1000 images réelles
   Objectif : générer 3040 images

  Epoch  20/100 | Loss D: 0.3722 | Loss G: 3.6036
  Epoch  40/100 | Loss D: 0.4483 | Loss G: 2.8305
  Epoch  60/100 | Loss D: 0.5621 | Loss G: 5.0643
  Epoch  80/100 | Loss D: 0.4434 | Loss G: 4.6760
  Epoch 100/100 | Loss D: 0.3246 | Loss G: 4.2748

  💾 Générateur sauvegardé !
  ✅ 3040 images générées dans /content/drive/MyDrive/projet_chiens/emotions/augmented/neutral

✅ neutral terminé !

Label : frown | À générer : 2248 images

🔧 GAN pour 'frown' — 1792 images réelles
   Objectif : générer 2248 images

  Epoch  20/100 | Loss D: 0.5930 | Loss G: 3.9163
  Epoch  40/100 | Loss D: 1.1362 | Loss G: 8.7805
  Epoch  60/100 | Loss D: 0.2600 | Loss G: 4.2187
  Epoch  80/100 | Loss D: 0.2513 | Loss G: 3.4714
  Epoch 100/100 | Loss D: 0.1744 | Loss G: 4.9718

  💾 Générateur sauvegardé !
 

Le DCGAN a été entraîné séparément pour les 3 classes minoritaires (neutral, frown, alert). Les valeurs de Loss G élevées s'expliquent par le petit nombre d'images d'entraînement — ce déséquilibre GAN classique est attendu avec moins de 2000 images. Malgré cela, les images générées apportent la variété nécessaire pour équilibrer le dataset.

In [ ]:
# ============================================================
# VÉRIFICATION — Contenu actuel de AUG_DIR
# ============================================================
import os

print("📂 Contenu de AUG_DIR :\n")

total = 0
for label in LABEL_TO_GROUPE.keys():
    dossier = os.path.join(AUG_DIR, label)
    if os.path.exists(dossier):
        n = len([f for f in os.listdir(dossier)
                 if f.lower().endswith('.jpg')])
        status = "✅" if n > 0 else "❌"
        print(f"  {status} {label:10s} → {n} images")
        total += n
    else:
        print(f"  ❌ {label:10s} → absent")

print(f"\n  TOTAL : {total} images")

📂 Contenu de AUG_DIR :

  ✅ happy      → 4040 images
  ✅ relaxed    → 4040 images
  ✅ neutral    → 4040 images
  ✅ sad        → 3644 images
  ✅ frown      → 2248 images
  ✅ alert      → 2177 images
  ❌ angry      → absent

  TOTAL : 20189 images


In [ ]:
# ============================================================
# CELLULE 12 — Undersampling + Fusion dans AUG_DIR
# ============================================================
# On équilibre le dataset :
#   → classes > TARGET : on undersample à TARGET
#   → classes < TARGET : on copie les originaux
#     + images GAN déjà générées dans AUG_DIR
#
# Pourquoi undersampler AVANT le split ?
#   → dataset équilibré → split équilibré ✅
#   → pas de biais dans train/val/test ✅
#.
# TARGET = 4040 images par label
# ============================================================
import random
import shutil

random.seed(SEED)

print(f"🎯 TARGET : {TARGET} images par label\n")

for label, chemins in images_par_label.items():
    dest_dir = os.path.join(AUG_DIR, label)
    os.makedirs(dest_dir, exist_ok=True)

    n_orig = len(chemins)

    if n_orig >= TARGET:
        # Undersampling → garder TARGET images originales
        garder = random.sample(chemins, TARGET)
        for i, fpath in enumerate(garder):
            fname = f"{label}_{i:05d}.jpg"
            dst   = os.path.join(dest_dir, fname)
            shutil.copy2(fpath, dst)
        print(f"  ✂️  {label:10s} {n_orig:5d} → {TARGET} "
              f"(undersampling)")
    else:
        # Copier tous les originaux
        for i, fpath in enumerate(chemins):
            fname = f"{label}_{i:05d}.jpg"
            dst   = os.path.join(dest_dir, fname)
            shutil.copy2(fpath, dst)

        # Les images GAN sont déjà dans AUG_DIR/label
        n_gan = len([f for f in os.listdir(dest_dir)
                     if f.endswith('.jpg') and 'gan' in f])
        n_total = len([f for f in os.listdir(dest_dir)
                       if f.endswith('.jpg')])
        print(f"  🔧 {label:10s} {n_orig:5d} + {n_gan} GAN "
              f"= {n_total} images")

print(f"\n✅ Fusion terminée !")
print(f"📁 Résultat dans : {AUG_DIR}")

🎯 TARGET : 4040 images par label

  ✂️  happy       7034 → 4040 (undersampling)
  ✂️  relaxed     5687 → 4040 (undersampling)
  🔧 neutral     1000 + 3040 GAN = 4040 images


KeyboardInterrupt: 

In [ ]:
# ============================================================
# REPRISE — Compléter AUG_DIR depuis où on s'est arrêté
# ============================================================
import random, shutil, os

random.seed(SEED)
TARGET = 4040

# Recharger images_par_label depuis CLEAN_DIR
images_par_label = {}
for label in LABEL_TO_GROUPE.keys():
    dossier = os.path.join(CLEAN_DIR, label)
    if os.path.exists(dossier):
        chemins = sorted([
            os.path.join(dossier, f)
            for f in os.listdir(dossier)
            if f.lower().endswith('.jpg')
        ])
        images_par_label[label] = chemins

print(f"🎯 TARGET : {TARGET} images par label\n")

for label, chemins in images_par_label.items():
    dest_dir = os.path.join(AUG_DIR, label)
    os.makedirs(dest_dir, exist_ok=True)

    # Compter ce qui existe déjà
    n_existe = len([f for f in os.listdir(dest_dir)
                    if f.endswith('.jpg')])

    if n_existe >= TARGET:
        print(f"  ⏭️  {label:10s} → déjà {n_existe} images ✅")
        continue

    n_orig = len(chemins)

    if n_orig >= TARGET:
        # Undersampling
        garder = random.sample(chemins, TARGET)
        for i, fpath in enumerate(garder):
            fname = f"{label}_{i:05d}.jpg"
            dst   = os.path.join(dest_dir, fname)
            shutil.copy2(fpath, dst)
        print(f"  ✂️  {label:10s} {n_orig} → {TARGET} (undersampling)")
    else:
        # Copier les originaux
        for i, fpath in enumerate(chemins):
            fname = f"{label}_{i:05d}.jpg"
            dst   = os.path.join(dest_dir, fname)
            if not os.path.exists(dst):
                shutil.copy2(fpath, dst)

        n_total = len([f for f in os.listdir(dest_dir)
                       if f.endswith('.jpg')])
        n_gan   = len([f for f in os.listdir(dest_dir)
                       if f.endswith('.jpg') and 'gan' in f])
        print(f"  🔧 {label:10s} {n_orig} + {n_gan} GAN = {n_total}")

print(f"\n✅ Fusion terminée !")

🎯 TARGET : 4040 images par label

  ⏭️  happy      → déjà 4040 images ✅
  ⏭️  relaxed    → déjà 4040 images ✅
  ⏭️  neutral    → déjà 4040 images ✅
  ✂️  sad        6244 → 4040 (undersampling)
  🔧 frown      1792 + 2248 GAN = 4040
  🔧 alert      1863 + 2177 GAN = 4040
  ✂️  angry      4665 → 4040 (undersampling)

✅ Fusion terminée !


In [ ]:
# ============================================================
# RECHARGEMENT — Depuis AUG_DIR (tout est prêt)
# ============================================================
images_par_label = {}

print("📂 Chargement depuis AUG_DIR...\n")

for label in LABEL_TO_GROUPE.keys():
    dossier = os.path.join(AUG_DIR, label)
    chemins = sorted([
        os.path.join(dossier, f)
        for f in os.listdir(dossier)
        if f.lower().endswith('.jpg')
    ])
    images_par_label[label] = chemins
    groupe = LABEL_TO_GROUPE[label]
    print(f"  ✅ {label:10s} → {groupe:7s} → {len(chemins):5d} images")

total  = sum(len(v) for v in images_par_label.values())
TARGET = 4040
print(f"\n  TOTAL  : {total} images")
print(f"  TARGET : {TARGET} images par label")

📂 Chargement depuis AUG_DIR...

  ✅ happy      → normal  →  4040 images
  ✅ relaxed    → normal  →  4040 images
  ✅ neutral    → normal  →  4040 images
  ✅ sad        → normal  →  4040 images
  ✅ frown      → normal  →  4040 images
  ✅ alert      → anormal →  4040 images
  ✅ angry      → anormal →  4040 images

  TOTAL  : 28280 images
  TARGET : 4040 images par label


In [ ]:
# ============================================================
# CELLULE 13 — Split 80/10/10
# ============================================================
# On divise le dataset en 3 parties :
#   train → 80% → apprentissage du modèle
#   val   → 10% → validation pendant l'entraînement
#   test  → 10% → évaluation finale
#
# Pourquoi ce split ?
#   → standard en Deep Learning ✅
#   → assez de données pour train ✅
#   → val et test représentatifs ✅
#
# Important : le split se fait APRÈS l'augmentation
#   → les images GAN sont dans train uniquement ✅
#   → val et test contiennent seulement
#     des images réelles ✅
# ============================================================
import random, shutil, os

random.seed(SEED)

# Créer les dossiers
for split in ['train', 'val', 'test']:
    for label in LABEL_TO_GROUPE.keys():
        os.makedirs(os.path.join(FINAL_DIR, split, label),
                    exist_ok=True)

print("📊 Split 80/10/10 en cours...\n")

for label, chemins in images_par_label.items():

    # Séparer originaux et GAN
    originaux  = [f for f in chemins if 'gan' not in os.path.basename(f)]
    gan_imgs   = [f for f in chemins if 'gan' in os.path.basename(f)]

    # Mélanger les originaux
    random.shuffle(originaux)

    n       = len(originaux)
    n_train = int(n * 0.80)
    n_val   = int(n * 0.10)

    splits = {
        'train' : originaux[:n_train] + gan_imgs,  # originaux + GAN
        'val'   : originaux[n_train:n_train + n_val],  # originaux seulement
        'test'  : originaux[n_train + n_val:]          # originaux seulement
    }

    for split, files in splits.items():
        for i, fpath in enumerate(files):
            ext   = os.path.splitext(fpath)[1]
            fname = f"{label}_{split}_{i:05d}{ext}"
            dst   = os.path.join(FINAL_DIR, split, label, fname)
            shutil.copy2(fpath, dst)

    print(f"  ✅ {label:10s} → "
          f"train:{len(splits['train']):4d} | "
          f"val:{len(splits['val']):4d} | "
          f"test:{len(splits['test']):4d}")

print(f"\n✅ Split terminé !")
print(f"📁 Résultat dans : {FINAL_DIR}")

📊 Split 80/10/10 en cours...

  ✅ happy      → train:3232 | val: 404 | test: 404
  ✅ relaxed    → train:3232 | val: 404 | test: 404
  ✅ neutral    → train:3840 | val: 100 | test: 100
  ✅ sad        → train:3232 | val: 404 | test: 404
  ✅ frown      → train:3681 | val: 179 | test: 180
  ✅ alert      → train:3667 | val: 186 | test: 187
  ✅ angry      → train:3232 | val: 404 | test: 404

✅ Split terminé !
📁 Résultat dans : /content/drive/MyDrive/projet_chiens/emotions/final


Le split 80/10/10 présente un déséquilibre dans les ensembles de validation et de test pour certaines classes comme neutral (100 images), alert (186 images) et frown (179 images). Ce déséquilibre est une conséquence directe de notre choix méthodologique de réserver les images synthétiques générées par DCGAN exclusivement à l'ensemble d'entraînement. Cette décision suit les bonnes pratiques en apprentissage automatique : évaluer le modèle uniquement sur des données réelles garantit que les métriques de performance (accuracy, recall, F1-score) reflètent fidèlement le comportement du modèle face à des cas réels. Introduire des images GAN dans val/test aurait artificiellement gonflé les scores d'évaluation sans refléter la réalité

In [ ]:
# ============================================================
# CELLULE 14 — Génération labels.csv
# ============================================================
# On crée un fichier CSV avec :
#   filepath → chemin relatif de l'image
#   label    → label original (happy, angry, alert...)
#   groupe   → normal ou anormal
#   split    → train, val ou test
#
# Ce fichier est essentiel pour la Late Fusion
# car il contient le mapping label → groupe
# ============================================================
import pandas as pd

print("📝 Génération labels.csv...\n")

records = []

for split in ['train', 'val', 'test']:
    for label in LABEL_TO_GROUPE.keys():
        label_dir = os.path.join(FINAL_DIR, split, label)
        if not os.path.exists(label_dir):
            continue
        for fname in os.listdir(label_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                records.append({
                    'filepath' : os.path.join(split, label, fname),
                    'label'    : label,
                    'groupe'   : LABEL_TO_GROUPE[label],
                    'split'    : split
                })

df = pd.DataFrame(records)
df.to_csv(CSV_PATH, index=False)

print(f"✅ labels.csv → {len(df)} lignes\n")
print("📊 Distribution par groupe :")
print(df.groupby(['split', 'groupe']).size().unstack(fill_value=0))
print("\n📊 Distribution par label :")
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))
print(f"\n📁 Sauvegardé dans : {CSV_PATH}")

📝 Génération labels.csv...

✅ labels.csv → 28280 lignes

📊 Distribution par groupe :
groupe  anormal  normal
split                  
test        591    1492
train      6899   17217
val         590    1491

📊 Distribution par label :
label  alert  angry  frown  happy  neutral  relaxed   sad
split                                                    
test     187    404    180    404      100      404   404
train   3667   3232   3681   3232     3840     3232  3232
val      186    404    179    404      100      404   404

📁 Sauvegardé dans : /content/drive/MyDrive/projet_chiens/emotions/labels.csv


In [ ]:
# ============================================================
# CELLULE 15 — ZIP final
# ============================================================
# On compresse le dossier final pour l'uploader sur Kaggle
# Le ZIP contient :
#   → train/ val/ test/ avec les 7 labels
#   → prêt pour l'entraînement des modèles
# ============================================================
import shutil, os

ZIP_PATH = f'{BASE}/emotions_final'

print("📦 Compression en cours...")

shutil.make_archive(
    ZIP_PATH,   # nom du zip
    'zip',      # format
    BASE,       # dossier parent
    'final'     # dossier à zipper
)

size = os.path.getsize(ZIP_PATH + '.zip')
print(f"✅ ZIP créé : {ZIP_PATH}.zip")
print(f"📁 Taille   : {size / (1024*1024):.1f} MB")

📦 Compression en cours...
✅ ZIP créé : /content/drive/MyDrive/projet_chiens/emotions/emotions_final.zip
📁 Taille   : 531.0 MB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

CSV_PATH  = '/content/drive/MyDrive/projet_chiens/emotions/labels.csv'
FINAL_DIR = '/content/drive/MyDrive/projet_chiens/emotions/final'

# Charger et filtrer
df = pd.read_csv(CSV_PATH)
df_new = df[df['label'].isin(['happy', 'alert', 'angry'])].copy()

print(f"Avant : {len(df)} images")
print(f"Après : {len(df_new)} images")
print(f"\nDistribution :\n{df_new['label'].value_counts()}")
print(f"\nSplit :\n{df_new['split'].value_counts()}")

Avant : 28280 images
Après : 12120 images

Distribution :
label
happy    4040
alert    4040
angry    4040
Name: count, dtype: int64

Split :
split
train    10131
test       995
val        994
Name: count, dtype: int64


In [ ]:
import pandas as pd, shutil, json, os

CSV_PATH  = '/content/drive/MyDrive/projet_chiens/emotions/labels.csv'
NEW_CSV   = '/content/drive/MyDrive/projet_chiens/emotions/labels_3class.csv'
NEW_FINAL = '/content/drive/MyDrive/projet_chiens/emotions/final_3class'

# Créer labels_3class.csv
df = pd.read_csv(CSV_PATH)
df_new = df[df['label'].isin(['happy', 'alert', 'angry'])].copy()
df_new.to_csv(NEW_CSV, index=False)
print(f"✅ labels_3class.csv créé — {len(df_new)} images")

✅ labels_3class.csv créé — 12120 images


In [ ]:
import shutil, json, os

NEW_FINAL = '/content/drive/MyDrive/projet_chiens/emotions/final_3class'
NEW_CSV   = '/content/drive/MyDrive/projet_chiens/emotions/labels_3class.csv'

# Vérifier que final_3class existe
if os.path.exists(NEW_FINAL):
    print(f"✅ final_3class existe")
    for split in ['train', 'val', 'test']:
        print(f"  {split} : {os.listdir(f'{NEW_FINAL}/{split}')}")
else:
    print("❌ final_3class n'existe pas — il faut le recréer")

❌ final_3class n'existe pas — il faut le recréer


In [ ]:
import os, shutil

NEW_FINAL = '/content/drive/MyDrive/projet_chiens/emotions/final_3class'
OLD_FINAL = '/content/drive/MyDrive/projet_chiens/emotions/final'

# Créer la structure
for split in ['train', 'val', 'test']:
    for label in ['happy', 'alert', 'angry']:
        os.makedirs(f'{NEW_FINAL}/{split}/{label}', exist_ok=True)

# Copier seulement les 3 classes
for split in ['train', 'val', 'test']:
    for label in ['happy', 'alert', 'angry']:
        src = f'{OLD_FINAL}/{split}/{label}'
        dst = f'{NEW_FINAL}/{split}/{label}'
        for f in os.listdir(src):
            shutil.copy2(f'{src}/{f}', f'{dst}/{f}')
        print(f"✅ {split}/{label} → {len(os.listdir(dst))} images")

print("\n✅ Dossier final_3class créé !")

✅ train/happy → 3232 images
✅ train/alert → 3667 images
✅ train/angry → 3232 images
✅ val/happy → 404 images
✅ val/alert → 186 images
✅ val/angry → 404 images
✅ test/happy → 404 images
✅ test/alert → 187 images
✅ test/angry → 404 images

✅ Dossier final_3class créé !


In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_token = '{"username":"zeinebbayoudh7","key":"7292fda1d604489aeceec79b42ae8c44"}'

with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_token)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Kaggle API configurée")

✅ Kaggle API configurée


In [ ]:
import json

NEW_FINAL = '/content/drive/MyDrive/projet_chiens/emotions/final_3class'
NEW_CSV   = '/content/drive/MyDrive/projet_chiens/emotions/labels_3class.csv'

# Copier labels.csv
shutil.copy(NEW_CSV, f'{NEW_FINAL}/labels.csv')

# Créer metadata
meta = {
    "title": "emotions-3class",
    "id": "zeinebbayoudh7/emotions-3class",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(f'{NEW_FINAL}/dataset-metadata.json', 'w') as f:
    json.dump(meta, f)

print("✅ Metadata créé")

# Upload sur Kaggle
!kaggle datasets create -p {NEW_FINAL} --dir-mode zip

✅ Metadata créé
Starting upload for file train.zip
100% 195M/195M [00:05<00:00, 34.8MB/s]
Upload successful: train.zip (195MB)
Starting upload for file val.zip
100% 20.8M/20.8M [00:01<00:00, 13.6MB/s]
Upload successful: val.zip (21MB)
Starting upload for file test.zip
100% 20.9M/20.9M [00:01<00:00, 13.6MB/s]
Upload successful: test.zip (21MB)
Starting upload for file labels.csv
100% 626k/626k [00:00<00:00, 725kB/s]
Upload successful: labels.csv (626KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/zeinebbayoudh7/emotions-3class


In [ ]:
!kaggle datasets create -p /content/drive/MyDrive/projet_chiens/emotions/final_3class --dir-mode zip

Starting upload for file train.zip
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/blobs.BlobApiService/StartBlobUpload


upload sans ZIP via Kaggle API depuis Colab.

In [ ]:
!pip install kaggle -q

import os
os.makedirs('/root/.kaggle', exist_ok=True)

# Remplace par tes vraies valeurs du kaggle.json
kaggle_token = '{"username":"zeinebbayoudh7","key":"8f430118b64f3bde9978c769b6135b3c"}'

with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_token)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Kaggle API configurée")

✅ Kaggle API configurée


In [ ]:
import os, json

# Créer le metadata
meta = {
    "title": "emotions-final",
    "id": "zeinebbayoudh7/emotions-final",
    "licenses": [{"name": "CC0-1.0"}]
}

UPLOAD_DIR = '/content/drive/MyDrive/projet_chiens/emotions/final'

with open(f'{UPLOAD_DIR}/dataset-metadata.json', 'w') as f:
    json.dump(meta, f)

print("✅ Metadata créé")

# Upload sur Kaggle
!kaggle datasets version -p {UPLOAD_DIR} -m "upload direct sans zip"

✅ Metadata créé
Skipping folder: train; use '--dir-mode' to upload folders
Skipping folder: val; use '--dir-mode' to upload folders
Skipping folder: test; use '--dir-mode' to upload folders
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion


In [ ]:
!kaggle datasets create -p /content/drive/MyDrive/projet_chiens/emotions/final --dir-mode zip

Starting upload for file train.zip
100% 444M/444M [00:04<00:00, 94.8MB/s]
Upload successful: train.zip (444MB)
Starting upload for file val.zip
100% 42.8M/42.8M [00:00<00:00, 76.5MB/s]
Upload successful: val.zip (43MB)
Starting upload for file test.zip
100% 43.1M/43.1M [00:00<00:00, 77.8MB/s]
Upload successful: test.zip (43MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/zeinebbayoudh7/emotions-final


In [ ]:
# Copier labels.csv dans le dossier final puis mettre à jour le dataset
import shutil

shutil.copy(
    '/content/drive/MyDrive/projet_chiens/emotions/labels.csv',
    '/content/drive/MyDrive/projet_chiens/emotions/final/labels.csv'
)

# Mettre à jour le dataset Kaggle
!kaggle datasets version -p /content/drive/MyDrive/projet_chiens/emotions/final -m "add labels.csv" --dir-mode zip

print("✅ Done")

Starting upload for file train.zip
100% 444M/444M [00:05<00:00, 90.0MB/s]
Upload successful: train.zip (444MB)
Starting upload for file val.zip
100% 42.8M/42.8M [00:00<00:00, 79.4MB/s]
Upload successful: val.zip (43MB)
Starting upload for file test.zip
100% 43.1M/43.1M [00:00<00:00, 75.6MB/s]
Upload successful: test.zip (43MB)
Starting upload for file labels.csv
100% 1.44M/1.44M [00:00<00:00, 6.82MB/s]
Upload successful: labels.csv (1MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/zeinebbayoudh7/emotions-final
✅ Done
